# 🐴 Module 10: Django Async (Views, Async ORM, Channels)

Django began as a synchronous WSGI monolith, but has evolved into a powerful ASGI hybrid.

### What We Will Master:
1. **Async Views in Django**: `async def my_view(request)`.
2. **Django 4.2+ Async ORM**: `aget()`, `afirst()`, `acount()`, `async for` over QuerySets.
3. **The `SynchronousOnlyOperation` Exception**: Why Django throws it and how to solve it.
4. **`sync_to_async` & `async_to_sync`** (`asgiref`).
5. **Django Channels**: WebSockets and Real-Time Push.

In [1]:
import asyncio
import time

async def mock_django_orm_aget(model_id: int):
    print(f"[{time.strftime('%X')}] 🐴 Running async ORM query: aget(id={model_id})")
    await asyncio.sleep(0.1)
    return {"id": model_id, "username": "rabin_p"}

async def fetch_credit_score(user_id: int):
    print(f"[{time.strftime('%X')}] 🌐 Concurrently fetching external customer credit rating...")
    await asyncio.sleep(0.3)
    return 780

# Inside an async Django View:
print("Executing Django Async View Simulation...")
start = time.perf_counter()
user, score = await asyncio.gather(mock_django_orm_aget(42), fetch_credit_score(42))
print(f"✅ View returned HTTP 200 in {time.perf_counter() - start:.2f}s with user data and credit rating!")

Executing Django Async View Simulation...
[08:35:01] 🐴 Running async ORM query: aget(id=42)
[08:35:01] 🌐 Concurrently fetching external customer credit rating...
✅ View returned HTTP 200 in 0.31s with user data and credit rating!


---
## 2. Overcoming `SynchronousOnlyOperation` with `sync_to_async`

If you call a legacy sync Django method (like `user.profile.save()`) inside an async view, Django raises:
`SynchronousOnlyOperation: You cannot call this from an async context - use a thread or sync_to_async.`

Here is how to bridge it safely with `sync_to_async`:

In [2]:
def legacy_sync_save_user(user_id: int):
    time.sleep(0.1) # Legacy blocking database write
    return f"Profile {user_id} updated"

# Modern equivalent of asgiref.sync.sync_to_async:
async def async_safe_save(user_id: int):
    return await asyncio.to_thread(legacy_sync_save_user, user_id)

print("Wrapping legacy synchronous Django method with sync_to_async...")
res = await async_safe_save(42)
print("Legacy model method executed safely on worker thread:", res)
print("✅ View returned without raising SynchronousOnlyOperation!")

Wrapping legacy synchronous Django method with sync_to_async...
Legacy model method executed safely on worker thread: Profile 42 updated
✅ View returned without raising SynchronousOnlyOperation!
